# ⚫ KPI Engineering and Feature Creation

#Objective:
The objective of this phase is to generate meaningful analytical insights from the cleaned dataset by computing Key Performance Indicators (KPIs) that normalize military strength relative to economic and manpower factors. This step also enriches the dataset with essential metadata such as continent, region, and alliance classification to support advanced filtering, comparison, and interactive dashboard visualization.

#Step 1: Configuration and Load Dataset

Load the cleaned datset.

In [ ]:
import pandas as pd
import numpy as np

# STEP 1: Load dataset
input_path = "military_cleaned.csv"
df_clean = pd.read_csv(input_path)

print("Original shape:", df_clean.shape)
df_kpi = df_clean.copy()

Original shape: (145, 33)


#Step 2: Metadata Enrichment

Enrich the dataset by mapping each country to its continent,region,and alliance status.

In [ ]:
pip install pycountry pycountry_convert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 17.1 MB/s eta 0:00:00


In [ ]:

# FULL AUTOMATIC REGION & CONTINENT MAPPING
# !pip install pycountry pycountry_convert
import pycountry
import pycountry_convert as pc

# --- helper function ---
def get_continent(country_name):
    try:
        country_alpha2 = pycountry.countries.lookup(country_name).alpha_2
        continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
        continent_name = pc.convert_continent_code_to_continent_name(continent_code)
        return continent_name
    except:
        return "Other"

# Apply continent mapping
df_kpi["continent"] = df_kpi["country"].apply(get_continent)

# REGION = same as continent (project standard)
df_kpi["region"] = df_kpi["continent"]

# NATO FLAG (correct list)
nato_countries = {
    "United States", "United Kingdom", "France", "Germany", "Italy",
    "Canada", "Spain", "Netherlands", "Belgium", "Turkey", "Poland",
    "Norway", "Denmark", "Portugal", "Greece", "Czech Republic",
    "Hungary", "Romania", "Bulgaria", "Slovakia", "Slovenia",
    "Estonia", "Latvia", "Lithuania", "Croatia", "Albania",
    "Montenegro", "North Macedonia", "Finland", "Sweden"
}

df_kpi["alliance_flag"] = df_kpi["country"].apply(
    lambda x: "NATO" if x in nato_countries else "Non-NATO"
)

# FINAL CLEANUP
df_kpi["continent"] = df_kpi["continent"].fillna("Other")
df_kpi["region"] = df_kpi["region"].fillna("Other")
df_kpi["alliance_flag"] = df_kpi["alliance_flag"].fillna("Non-NATO")

# Check
print("Missing values:")
print(df_kpi[["continent","region","alliance_flag"]].isna().sum())

print("\nUnique continents found:")
print(df_kpi["continent"].unique())

Missing values:
continent        0
region           0
alliance_flag    0
dtype: int64

Unique continents found:
['Asia' 'South America' 'North America' 'Africa' 'Other' 'Europe'
 'Oceania']


#Step 3: KPI Calculations

Compute KPIs such as total military hardware,assets per capita ,budget _to_gdp ratio,and power index rank gap.

In [ ]:
# Convert numerics safely
def to_num(col):
    if col in df_kpi.columns:
        df_kpi[col] = pd.to_numeric(df_kpi[col], errors="coerce")

for col in [
    "total_aircraft",
    "tanks",
    "naval_assets",
    "total_manpower",
    "defense_budget",
    "gdp"
]:
    to_num(col)

# Prevent divide-by-zero
if "total_manpower" in df_kpi.columns:
    df_kpi["total_manpower"] = df_kpi["total_manpower"].replace(0, np.nan)

if "gdp" in df_kpi.columns:
    df_kpi["gdp"] = df_kpi["gdp"].replace(0, np.nan)

# Assets per Capita
def col_or_zero(name):
    return df_kpi[name] if name in df_kpi.columns else 0

df_kpi["assets_per_capita"] = (
    col_or_zero("total_aircraft")
    + col_or_zero("tanks")
    + col_or_zero("naval_assets")
) / col_or_zero("total_manpower")

# Budget to GDP Ratio
df_kpi["budget_to_gdp_ratio"] = (
    col_or_zero("defense_budget") / col_or_zero("gdp")
)



#Step 4: Standardize Column Names

Standardize all column names using consistent lower_case snake_case formatting.

In [ ]:
# Standardize column names
df_kpi.columns = (
    df_kpi.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nAvailable columns:")
print(df_kpi.columns.tolist())



Available columns:
['country', 'gdp', 'defense_budget', 'total_manpower', 'active_personnel', 'reserve_personnel', 'air_force_personnel', 'army_personnel', 'navy_personnel', 'total_aircraft', 'attack_aircraft', 'fighter_aircraft', 'transport_aircraft', 'helicopters', 'special_mission_aircraft', 'trainer_aircraft', 'attack_helicopters', 'tanker_aircraft', 'naval_assets', 'aircraft_carriers', 'helicopter_carriers', 'destroyers', 'frigates', 'corvettes', 'submarines', 'patrol_vessel', 'mine_warfare', 'tanks', 'self_propelled_artillery', 'towed_artillery', 'rocket_artillery', 'external_debt', 'power_index', 'assets_per_capita', 'budget_to_gdp_ratio']


#Step 5: Validate KPI Values

Validate KPI calculations and handle any infinite or missing values.

In [ ]:
# CREATE RANK FROM POWER INDEX
if "power_index" in df_kpi.columns:

    # ensure numeric
    df_kpi["power_index"] = pd.to_numeric(
        df_kpi["power_index"],
        errors="coerce"
    )

    # create rank (lower index = better rank)
    df_kpi["power_index_rank"] = df_kpi["power_index"].rank(
        method="min",
        ascending=True
    )

    # compute gap from top country
    top_rank = df_kpi["power_index_rank"].min()

    df_kpi["power_index_rank_gap"] = (
        df_kpi["power_index_rank"] - top_rank
    )

    print("✅ Rank created from power_index")

else:
    print("❌ power_index column missing")
    df_kpi["power_index_rank_gap"] = np.nan


# Fix scientific notation
df_kpi["assets_per_capita"] = df_kpi["assets_per_capita"].round(8)
df_kpi["budget_to_gdp_ratio"] = df_kpi["budget_to_gdp_ratio"].round(8)


✅ Rank created from power_index


#Step 6: Export Final Dataset

Save completely cleaned dataset to a new final CSV file.

In [ ]:
# Export final file
output_path = "military_final.csv"

df_kpi.to_csv(
    output_path,
    index=False,
    float_format="%.8f"
)

print("\n✅ FINAL FILE CREATED SUCCESSFULLY")
print("Saved to:", output_path)



✅ FINAL FILE CREATED SUCCESSFULLY
Saved to: military_final.csv


#Takeaway:
Successfully calculated 3 KPIs assests_per_capita,budget_gdp_ratio,power_index_rank_gap across all 145 rows,exporting to military_final.csv.